
# Week 5: Semantic Search with Embeddings

## Objective

The goal of Week 5 is to build a semantic search system
for real estate listings using sentence embeddings and FAISS.

Traditional keyword search looks for exact or similar words.
Semantic search focuses on the meaning of a query, even when
the property description uses different wording.

For this project, I will:
1. Load real estate listing remarks from MySQL.
2. Generate 384-dimensional sentence embeddings.
3. Build a FAISS index for similarity search.
4. Compare semantic search with BM25 keyword search.
5. Measure search latency.
6. Evaluate the relevance of search results.

The main goal is to determine whether semantic search provides
more relevant results than traditional keyword matching.


In [1]:

# Import libraries used throughout the notebook.
# pandas helps us organize and display the results.
# Path helps us locate our project files.

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time

# Find the main project folder.
# Our notebook is inside notebooks/, so we may need
# to move one folder up to import scripts/.

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP



## Loading the Dataset

I am using the real_estate database in MySQL, specifically
the rets_property table.

The main feature for semantic search is L_Remarks, which
contains written descriptions of properties.

I will initially use 10,000 listings to build and evaluate
the search system before expanding to the full dataset.



In [2]:

import mysql.connector
import getpass

# Connect to our existing real estate database.
# getpass hides the password while we type it.

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=getpass.getpass("MySQL password: "),
    database="real_estate"
)

# Select property descriptions and useful information.
# We remove listings with missing or empty remarks.

query = """
SELECT
    L_City,
    L_SystemPrice,
    L_Remarks
FROM rets_property
WHERE L_Remarks IS NOT NULL
  AND TRIM(L_Remarks) <> ''
LIMIT 10000
"""

cursor = conn.cursor()

try:
    cursor.execute(query)

    rows = cursor.fetchall()
    columns = cursor.column_names

    df = pd.DataFrame(rows, columns=columns)

finally:
    cursor.close()
    conn.close()

# Reset the row numbers so they match the FAISS positions.
df = df.reset_index(drop=True)

print("Total listings:", len(df))
display(df.head())


Total listings: 10000


,L_City,L_SystemPrice,L_Remarks
0,Beverly Hills,3950000,Opportunity to reimagine a Classic 70's Archit...
1,Lathrop,775000,Step into comfort and functionality with this ...
2,Sherman Oaks,750000,WOW! NOW REDUCED AND THE SELLER MAY FINANCE! C...
3,Mountain House,1122426,Plan 3 features an open floor plan with covere...
4,Mountain House,1128292,Plan 2 features an open floor plan with covere...



## Sentence Embeddings and FAISS

I will use the all-MiniLM-L6-v2 model to convert each
property description into a 384-dimensional vector.

The model represents text numerically so descriptions
with similar meanings can be compared.

FAISS stores these vectors and helps retrieve the most
similar listings efficiently.


In [3]:

from scripts.semantic_search import SemanticSearcher

# Create the semantic search model.
searcher = SemanticSearcher()

# Get property descriptions as a Python list.
remarks = df["L_Remarks"].fillna("").tolist()

# Generate embeddings and add them to FAISS.
# This may take several minutes for 10,000 listings.
searcher.build_index(remarks)

print("Indexed listings:", searcher.index.ntotal)
print("Embedding dimensions:", searcher.index.d)


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.